# Demo of pig detection and tracking models (Kaggle)
This Kaggle notebook demonstrates the capabilities of the pig detection and tracking models presented in the paper "Benchmarking pig detection and tracking under diverse and challenging conditions".

To run this notebook, you **need to enable a GPU accelerator**. Go to **Settings → Accelerator → GPU T4 x2** (or P100) on the right-hand panel.

**Before running**, you need to upload the model weights as a Kaggle Dataset:
1. Download `codino_swin.pth` from: https://data.goettingen-research-online.de/dataset.xhtml?persistentId=doi:10.25625/I6UYE9
2. Go to **kaggle.com → Datasets → New Dataset**, name it `pig-detect`, and upload `codino_swin.pth`.
3. In this notebook, click **+ Add Input** (right panel) → find your `pig-detect` dataset and add it.
   - The checkpoint will be available at `/kaggle/input/pig-detect/codino_swin.pth`.

# Create environment

Install the required packages. This takes roughly 4–6 minutes. Torch and torchvision are pinned to versions compatible with mmdet 3.3.0. After installation, **restart the kernel** (Run → Restart & Clear Output) and then run all cells again from the beginning.

In [ ]:
!pip install torch==2.0.0 torchvision==0.15.1 --index-url https://download.pytorch.org/whl/cu118 -q
!pip install -U openmim -q
!mim install mmengine -q
!mim install "mmcv==2.0.0" -q
!mim install mmdet==3.3.0 -q
!mim install mmyolo==0.6.0 -q
!pip install numpy==1.26.4 munch loguru ftfy lap filterpy -q
print("Installation complete!")

# Clone repo and load packages
After the environment has been set up, we clone the PigBench repository and load the necessary packages.

In [ ]:
!git clone https://github.com/huanshaolin/PigBench

In [ ]:
%cd PigBench/detection/tools/inference
import mmcv
import os
import sys
from mmdet.utils import register_all_modules
from mmdet.apis import init_detector, inference_detector
from visualization_utils import draw_bboxes
sys.path.append('../../')
# Ignore the message "Disabling PyTorch because..."

# Detection demo

Make sure you have added your `pig-detect` Kaggle dataset as an input (see instructions at the top of this notebook). The model checkpoint will be loaded from `/kaggle/input/pig-detect/codino_swin.pth`.

In [ ]:
# Config and checkpoint file paths
config_path = '../../configs/co_detr/co_dino_swin.py'
checkpoint_path = '/kaggle/input/pig-detect/codino_swin.pth'

register_all_modules(init_default_scope=False)
model = init_detector(config_path, checkpoint_path, device='cuda:0')

The next block runs inference on an example image from the repo. You can also upload your own pig images to your `pig-detect` Kaggle dataset and update `image_path` accordingly.

In [ ]:
# Run inference on a single example image
image_path = '../../data/example_images/demo_image1.jpg'

# Alternatively, use your own image uploaded to the pig-detect Kaggle dataset:
# image_path = '/kaggle/input/pig-detect/name_of_your_image.jpg'

image = mmcv.imread(image_path, channel_order='rgb')
result = inference_detector(model, image)

scores0 = result.pred_instances.scores.cpu().numpy()
bboxes0 = result.pred_instances.bboxes.cpu().numpy()

The next block visualizes the predicted bounding boxes and displays the total pig count.

In [ ]:
# score_thresh is the minimum score of a bbox that is required for it to be visualized
draw_bboxes(image_path=image_path, bboxes=bboxes0, scores=scores0,
            score_thresh=0.4, save_path=None, linewidth=2, show_scores=True, figsize=(15, 15), score_fontsize=6)

# Tracking demo

To run tracking, you need to add a video to your Kaggle dataset:
1. Download `PigTrackVideos.zip` from https://data.goettingen-research-online.de/dataset.xhtml?persistentId=doi:10.25625/P7VQTP and extract `pigtrack0001.mp4`.
2. Upload `pigtrack0001.mp4` to your `pig-detect` Kaggle dataset inside a folder named `example_videos`.
   - The video will be available at `/kaggle/input/pig-detect/example_videos/pigtrack0001.mp4`.

The Co-DINO model is quite slow, so inference time for the 14-second video is roughly 2–3 minutes on Kaggle GPU.

The tracking output (mp4 visualization) will be saved to `/kaggle/working/example_videos_results`.

In [ ]:
%cd /kaggle/working/PigBench/tracking/boxmot

In [ ]:
!python main.py \
  --config configs/botsort.yaml \
  --inference_detector_checkpoint /kaggle/input/pig-detect/codino_swin.pth \
  --seq_dir /kaggle/input/pig-detect/example_videos \
  --outputs_base /kaggle/working/example_videos_results

After the tracking run completes, the result video can be found in the **Output** tab on the right, under `/kaggle/working/example_videos_results`. You can download it from there.